# 03 · Promotion Readiness Assessment — Logistic Regression

**Goal** — produce an objective **promotion-readiness score** from performance history
(`performance_score`, `performance_last_year`, `manager_rating`), training completions
(`training_hours_last_year`, `mentoring_sessions`), certifications (`certifications_count`,
`skill_assessment_score`) and tenure (`years_at_company`, `years_in_current_role`,
`years_since_last_promotion`).

**Algorithm — Logistic Regression** (`LogisticRegression`). Chosen because promotion is a
high-stakes, must-be-defensible decision: logistic regression yields a calibrated probability
and transparent, auditable coefficients (odds ratios) HR can explain to employees — exactly
what an "objective readiness score" needs. Rationale + citations in
`../MODEL-JUSTIFICATION.md`. Binary classification on `employee_promotion_prediction.csv`
(~100k rows, ~10% promoted — strongly imbalanced).

> **Leakage note:** `salary_increase_percent` is dropped — a raise is granted *as part of* a
> promotion, so it would leak the outcome.

Logged to `logs/promotion*.log`; artifacts under `artifacts/promotion/`.

In [ ]:
# --- environment & logged run -------------------------------------------------------
import sys
from pathlib import Path

# notebooks/ live one level below the model root where synapse_ml.py sits
HERE = Path.cwd()
MODEL_DIR = HERE if (HERE / "synapse_ml.py").exists() else HERE.parent
sys.path.insert(0, str(MODEL_DIR))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)

import synapse_ml as sm
run = sm.start_run("promotion")   # opens logs/promotion_<ts>.log + artifacts/promotion/
log = run.log

## 1 · Load data

In [ ]:
df = run.load_csv("employee_promotion_prediction.csv")
print(df.shape)
df.head()

In [ ]:
import io
_info = io.StringIO()
df.info(buf=_info)
log.info("dataframe info:\n%s", _info.getvalue())
df.describe().T

## 2 · Target & exploratory analysis

In [ ]:
TARGET = "promoted"
ID_COL = "employee_id"
LEAKY = ["salary_increase_percent"]   # post-promotion artefact -> dropped
DROP = [ID_COL] + LEAKY

y = df[TARGET].astype(int)
rate = y.mean()
log.info("target=%s · positive rate=%.3f (%d of %d)", TARGET, rate, y.sum(), len(y))

fig, ax = plt.subplots(figsize=(4, 3))
y.value_counts().sort_index().plot(kind="bar", ax=ax, color=["#4c72b0", "#dd8452"])
ax.set_xticklabels(["Not promoted (0)", "Promoted (1)"], rotation=0)
ax.set_title(f"Promotion class balance · positive={rate:.1%}")
run.save_fig(fig, "01_class_balance"); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ["years_since_last_promotion", "department"]):
    rates = df.assign(_y=y).groupby(col)["_y"].mean().sort_values(ascending=False)
    log.info("promotion rate by %s:\n%s", col, rates.to_string())
    rates.plot(kind="bar", ax=ax, color="#8172b3")
    ax.set_title(f"Promotion rate by {col}"); ax.set_ylabel("rate")
    ax.tick_params(axis="x", rotation=45)
fig.tight_layout(); run.save_fig(fig, "02_promotion_by_driver"); plt.show()

In [ ]:
num_df = df.drop(columns=DROP + [TARGET]).select_dtypes("number").assign(_y=y)
corr = num_df.corr()["_y"].drop("_y").sort_values()
log.info("numeric correlation with promotion:\n%s", corr.to_string())

fig, ax = plt.subplots(figsize=(7, 8))
corr.plot(kind="barh", ax=ax, color=np.where(corr > 0, "#dd8452", "#4c72b0"))
ax.set_title("Correlation of numeric features with promotion")
run.save_fig(fig, "03_numeric_correlation"); plt.show()

## 3 · Preprocessing & stratified split

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = df.drop(columns=DROP + [TARGET])
numeric_cols, categorical_cols = sm.split_feature_types(df.drop(columns=DROP), target=TARGET)
log.info("numeric=%d · categorical=%d %s", len(numeric_cols), len(categorical_cols), categorical_cols)

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())]), numeric_cols),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
     categorical_cols),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
log.info("split · train=%s test=%s", X_train.shape, X_test.shape)

## 4 · Logistic Regression model

Scaling (done in preprocessing) matters for logistic regression's coefficients and
convergence. `class_weight="balanced"` handles the 10% positive rate. Read by 5-fold ROC-AUC.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=2000, C=1.0, class_weight="balanced")),
])

cv_auc = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
log.info("CV ROC-AUC = %.4f (+/- %.4f)", cv_auc.mean(), cv_auc.std())
model.fit(X_train, y_train)
log.info("fitted LogisticRegression on %d rows", len(X_train))

## 5 · Evaluation

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay,
                             PrecisionRecallDisplay)

proba = model.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)
roc = roc_auc_score(y_test, proba)
pr_auc = average_precision_score(y_test, proba)
report = classification_report(y_test, pred, target_names=["Not promoted", "Promoted"], digits=3)
log.info("test ROC-AUC=%.4f · PR-AUC=%.4f", roc, pr_auc)
log.info("classification report @0.5:\n%s", report)
print(report)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred),
                       display_labels=["Not", "Promoted"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix @ 0.5")
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[1])
axes[1].set_title(f"ROC (AUC={roc:.3f})"); axes[1].plot([0, 1], [0, 1], "k--", lw=0.8)
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[2])
axes[2].set_title(f"Precision-Recall (AP={pr_auc:.3f})")
fig.tight_layout(); run.save_fig(fig, "04_evaluation_curves"); plt.show()

In [ ]:
# Logistic-regression coefficients as odds ratios — the interpretability payoff.
ohe = model.named_steps["prep"].named_transformers_["cat"].named_steps["ohe"]
feat_names = numeric_cols + list(ohe.get_feature_names_out(categorical_cols))
coefs = pd.Series(model.named_steps["clf"].coef_[0], index=feat_names)
odds = np.exp(coefs).sort_values()
log.info("odds ratios (exp(coef)):\n%s", odds.to_string())

top = pd.concat([odds.head(8), odds.tail(8)])
fig, ax = plt.subplots(figsize=(7, 7))
top.plot(kind="barh", ax=ax, color=np.where(top > 1, "#dd8452", "#4c72b0"))
ax.axvline(1.0, color="k", lw=0.8)
ax.set_title("Promotion odds ratios · strongest drivers (>1 ↑, <1 ↓)")
run.save_fig(fig, "05_odds_ratios"); plt.show()

## 6 · Decision-threshold tuning

The shortlist is ranked, so we tune the threshold to maximise F1 on the "Promoted" class
rather than defaulting to 0.5.

In [ ]:
from sklearn.metrics import precision_recall_curve

prec, rec, thr = precision_recall_curve(y_test, proba)
f1s = 2 * prec * rec / (prec + rec + 1e-12)
best_idx = int(np.nanargmax(f1s[:-1]))
best_thr = float(thr[best_idx])
log.info("tuned threshold=%.3f · F1=%.3f · recall=%.3f · precision=%.3f",
         best_thr, f1s[best_idx], rec[best_idx], prec[best_idx])
print(classification_report(y_test, (proba >= best_thr).astype(int),
                            target_names=["Not promoted", "Promoted"], digits=3))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(thr, prec[:-1], label="precision"); ax.plot(thr, rec[:-1], label="recall")
ax.plot(thr, f1s[:-1], label="F1")
ax.axvline(best_thr, color="k", ls="--", lw=0.8, label=f"best={best_thr:.2f}")
ax.set_xlabel("threshold"); ax.set_title("Threshold sweep (Promoted class)"); ax.legend()
run.save_fig(fig, "06_threshold_sweep"); plt.show()

## 7 · Readiness scoring

Turn probabilities into a 0–100 **readiness score** and Low / Medium / High tiers, then
checkpoint the scored roster.

In [ ]:
scored = X_test.copy()
scored["readiness_score"] = (proba * 100).round(1)
scored["actual_promoted"] = y_test.values
scored["readiness_tier"] = pd.cut(proba, bins=[-0.01, 0.33, 0.66, 1.01],
                                  labels=["Low", "Medium", "High"])
log.info("readiness tier distribution:\n%s", scored["readiness_tier"].value_counts().sort_index().to_string())
ready = scored.sort_values("readiness_score", ascending=False).head(10)
log.info("top-10 promotion-ready:\n%s",
         ready[["readiness_score", "readiness_tier", "actual_promoted"]].to_string())
run.checkpoint_df(scored.reset_index(names="row_id"), "scored_readiness_roster")
ready[["readiness_score", "readiness_tier", "actual_promoted"]]

## 8 · Persist model & metrics

In [ ]:
run.save_model(model, "promotion_model")
run.save_metrics({
    "algorithm": "LogisticRegression",
    "cv_roc_auc": float(cv_auc.mean()),
    "test_roc_auc": roc, "test_pr_auc": pr_auc,
    "tuned_threshold": best_thr,
    "tuned_recall_promoted": float(rec[best_idx]),
    "tuned_precision_promoted": float(prec[best_idx]),
    "positive_rate": float(rate),
    "dropped_leaky_features": LEAKY,
    "high_readiness_count": int((scored["readiness_tier"] == "High").sum()),
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
})
run.finish(summary=f"LogisticRegression: test ROC-AUC={roc:.3f}, PR-AUC={pr_auc:.3f}, tuned thr={best_thr:.2f}")